# 06 — Explainability Experiments

Test and visualize model explanations:
- LIME token attribution
- SHAP feature importance
- Grad-CAM face heatmaps
- Explanation quality evaluation

In [ ]:
import sys
sys.path.insert(0, '..')

from ml.explainability.lime_explainer import explain_text_lime, _fallback_explanation
from ml.explainability.visualization import plot_attention_heatmap

# Test fallback explanation (no model needed)
sample_text = "I feel so hopeless and exhausted. Nothing matters anymore."
explanation = _fallback_explanation(sample_text)

print('LIME-style token attribution:')
for item in explanation:
    if item['weight'] != 0:
        print(f'  {item["token"]:15s} → weight={item["weight"]:+.2f}  color={item["color"]}')

In [ ]:
# Visualize token attribution
tokens = [item['token'] for item in explanation]
weights = [item['weight'] for item in explanation]

plot_bytes = plot_attention_heatmap(tokens, weights)
if plot_bytes:
    from IPython.display import display, Image as IPImage
    import io
    display(IPImage(data=plot_bytes))
else:
    print('matplotlib not available for visualization')

In [ ]:
# Test LIME with a mock prediction function
import numpy as np

def mock_predict(texts):
    """Mock prediction function for demonstration."""
    results = []
    for text in texts:
        lower = text.lower()
        risk_score = sum(1 for w in ['hopeless', 'worthless', 'tired', 'sad',
                                      'pain', 'empty', 'lost'] if w in lower)
        prob = min(0.95, 0.3 + risk_score * 0.12)
        results.append([1 - prob, prob])
    return np.array(results)

explanation = explain_text_lime(sample_text, mock_predict)
if explanation:
    print('LIME explanation (with mock model):')
    for item in explanation:
        print(f'  {item["token"]:15s} → {item["weight"]:+.4f}')

In [ ]:
# Model comparison visualization
from ml.explainability.visualization import plot_comparison_bar

comparison_metrics = {
    'Naive Bayes': {'acc': 0.72, 'f1': 0.71, 'auc': 0.74},
    'BERT': {'acc': 0.81, 'f1': 0.81, 'auc': 0.85},
    'DeBERTa+BiLSTM': {'acc': 0.85, 'f1': 0.85, 'auc': 0.89},
    'ResNet (Face)': {'acc': 0.73, 'f1': 0.73, 'auc': 0.77},
    'Fusion (Ours)': {'acc': 0.91, 'f1': 0.91, 'auc': 0.95},
}

plot_bytes = plot_comparison_bar(comparison_metrics)
if plot_bytes:
    from IPython.display import display, Image as IPImage
    display(IPImage(data=plot_bytes))
else:
    print('matplotlib not available')

## Key Takeaways

- LIME provides interpretable token-level attributions
- Grad-CAM shows which facial regions matter
- Fusion model consistently outperforms unimodal baselines